In [4]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt 
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, recall_score,
    precision_score, f1_score, cohen_kappa_score,
    matthews_corrcoef
)
from scipy.stats import chi2_contingency
from imblearn.over_sampling import RandomOverSampler
import kagglehub
import shutil
import os
from ucimlrepo import fetch_ucirepo

In [ ]:
# Kaggle
dataset_dir = "data"
if len(os.listdir(dataset_dir)) == 0:
    path = kagglehub.dataset_download("redwankarimsony/heart-disease-data")
    shutil.move(path, dataset_dir)

print("dataset downloaded")

# UCI dataset
# heart_disease = fetch_ucirepo(id=45) 
# X = heart_disease.data.features 
# y = heart_disease.data.targets 

In [ ]:
# uncomment if use data from kaggle
data_path = "data/6/heart_disease_uci.csv"
df = pd.read_csv(data_path, index_col=0)
print(df.columns)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isna().sum()

In [ ]:
df['ca'] = df['ca'].astype(str)
y = df['num']
X = df.drop(columns=['num', 'dataset'])
numerical_feature = X.select_dtypes(include=["int64", "float64"]).columns.to_list()
categorical_feature = X.select_dtypes(exclude=["int64", "float64"]).columns.to_list()
X[categorical_feature] = X[categorical_feature].astype(str)

In [ ]:
def transform_to_biner(x):
    if x != 0:
        x = 1
    return x

y = y.apply(transform_to_biner)

In [ ]:
imputer = SimpleImputer(strategy='mean')
X[numerical_feature] = imputer.fit_transform(X[numerical_feature])
X[categorical_feature] = X[categorical_feature].fillna("unknow")

X_bin = df[categorical_feature].copy()
for column in numerical_feature:
    X_bin[f"{column}_bin"] = pd.qcut(X[column], q=4, duplicates="drop")

X_bin.head()

In [ ]:
chi2_result = []
for column in X_bin:
    table = pd.crosstab(X_bin[column], y)
    chi2, p, _, _ = chi2_contingency(table)

    results = {
        "name": column,
        "chi2" : chi2,
        "p" : p 
    }

    chi2_result.append(results)

In [ ]:
result = pd.DataFrame(chi2_result)
result = result.sort_values("chi2", ascending=False)

print(result)